# Convolutions and Backpropagation: Under the Hood - Solutions

<hr style="clear:both">

This notebook is part of a series of exercises for the CIVIL-226 Introduction to Machine Learning for Engineers course at EPFL. Copyright (c) 2021 [VITA](https://www.epfl.ch/labs/vita/) lab at EPFL  
Use of this source code is governed by an MIT-style license that can be found in the LICENSE file or at https://www.opensource.org/licenses/MIT

**Author(s):** [Reyhaneh Hosseininejad](mailto:reyhaneh.hosseininejad@epfl.ch)
<hr style="clear:both">


Before diving into PyTorch and building deep Convolutional Neural Networks (CNNs), it is crucial to understand exactly what happens inside the "black box". 

In this notebook, we will manually implement the forward and backward passes of a tiny CNN using only `numpy`. You will see exactly how a convolution operation works, how gradients are routed backwards through pooling layers, and how weights are updated using the chain rule.

By the end of this exercise, you will be able to manually tweak parameters and observe the direct mathematical impact on the network.

## 1. Imports

In [ ]:
import numpy as np

# Set print options for clean output
np.set_printoptions(precision=4, suppress=True)

## 2. Setting up the Data and Filter
We will use a miniature 3x3 grayscale image patch and a single 2x2 convolutional filter. This is small enough to trace by hand but large enough to demonstrate the sliding window concept.

In [ ]:
# A simple 3x3 grayscale image patch (X)
X = np.array([
    [1.0, 2.0, 1.0],
    [0.0, 1.0, 3.0],
    [2.0, 0.0, 1.0]
])

# A 2x2 convolutional filter (W), initialized randomly
W = np.array([
    [0.5, -0.2],
    [-0.5, 0.8]
])

print("Input Image (X):\n", X)
print("\nFilter Weights (W):\n", W)

## 3. The Forward Pass
### 3.1 Convolution
A convolution involves sliding the filter `W` over the image `X`. At each step, we compute the element-wise multiplication between the filter and the image patch, and sum the results.
*Note: For simplicity, we are omitting the bias term in this convolution.*

**Task:** Complete the convolution operation below for a stride of 1 and no padding.

In [ ]:
out_conv = np.zeros((2, 2))

for i in range(2):
    for j in range(2):
        # Extract the 2x2 patch from the image X
        patch = X[i:i+2, j:j+2]
        
        ### START CODE HERE ###
        # Compute the element-wise multiplication and sum
        out_conv[i, j] = np.sum(patch * W)
        ### END CODE HERE ###

print("Feature Map after Convolution:\n", out_conv)

### 3.2 Activation (ReLU) and Max Pooling
Next, we apply the Rectified Linear Unit (ReLU) to introduce non-linearity, replacing negative values with zero. Then, we apply a 2x2 Max Pooling operation to extract the most prominent feature.

In [ ]:
### START CODE HERE ###
# Apply ReLU (hint: np.maximum)
out_relu = np.maximum(0, out_conv)

# Apply Max Pooling to the entire 2x2 relu output (hint: np.max)
out_pool = np.max(out_relu)
### END CODE HERE ###

print("Feature Map after ReLU:\n", out_relu)
print("\nPrediction after Max Pooling:", out_pool)

### 3.3 Computing the Loss
Let's assume our target prediction for this specific image patch is `5.0`. We will calculate the Mean Squared Error (MSE) loss. We multiply the loss by 0.5 so that the derivative simplifies perfectly to `(out_pool - target)` without a factor of 2.

In [ ]:
target = 5.0

# MSE Loss (multiplied by 0.5 to simplify the derivative later)
loss = 0.5 * (out_pool - target)**2
print("Loss:", loss)

## 4. The Backward Pass (Backpropagation)
In PyTorch, you simply call `loss.backward()`. Here, we will compute it manually using the **Chain Rule**. We need to find the gradient of the loss with respect to our filter weights: `dL/dW`.

### 4.1 Gradients through Pooling and ReLU
When routing gradients backward:
1. **Max Pool** only passes the gradient to the specific pixel that was the "maximum" during the forward pass. All other pixels get a gradient of 0.
2. **ReLU** passes the gradient through only if the forward pass value was > 0.

In [ ]:
# 1. Derivative of Loss with respect to the pooled output
dL_dpool = (out_pool - target)
print("Gradient from Loss (dL_dpool):", dL_dpool)

# 2. Routing gradient through Max Pool
dL_drelu = np.zeros((2, 2))
max_idx = np.unravel_index(np.argmax(out_relu), out_relu.shape)
dL_drelu[max_idx] = dL_dpool
print("\nGradient after backward Max Pool (dL_drelu):\n", dL_drelu)

# 3. Routing gradient through ReLU
dL_dconv = dL_drelu.copy()
dL_dconv[out_conv <= 0] = 0
print("\nGradient after backward ReLU (dL_dconv):\n", dL_dconv)

### 4.2 Gradient of the Weights (`dL/dW`)
To find how much each weight contributed to the error, we convolve the original input `X` with the incoming gradient `dL_dconv`. This is because the gradient of a convolution with respect to its weights is mathematically another convolution (between the inputs and the incoming gradient).

In [ ]:
dL_dW = np.zeros((2, 2))

for i in range(2):
    for j in range(2):
        patch = X[i:i+2, j:j+2]
        # Accumulate the gradients for the weights
        dL_dW += patch * dL_dconv[i, j]

print("Gradient of the Weights (dL_dW):\n", dL_dW)

## 5. The Optimizer Step
This represents `optimizer.step()` in PyTorch. We update the weights by moving them in the opposite direction of the gradient, scaled by the learning rate.

In [ ]:
learning_rate = 0.05

### START CODE HERE ###
# Update the weights using Gradient Descent
W_new = W - (learning_rate * dL_dW)
### END CODE HERE ###

print("Old Weights:\n", W)
print("\nNew Updated Weights:\n", W_new)

## 6. Playground
**Experimentation task:** Now that you have seen the full cycle, go back to Section 2 and change the values of the initial image `X` or weights `W`. 
- What happens to the gradients if the initial weights are all zeros?
- Try changing the `target` in Section 3.3. How does this affect `dL_dW`?
- Modify the `learning_rate`. What happens if it is too high (e.g., `1.5`)?
- **Bonus / Optional Challenge:** How could you compute the forward pass convolution using vectorized NumPy operations instead of nested for-loops?